In [ ]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [2]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['falkonml_falkon', 'juliamanifolds_manopt.jl', 'neuroml_pyneuroml', 'henriqueslab_zerocostdl4mic',\
           'frankyeh_dsi-studio', 'andreatramacere_jetset', 'micycle1_tinfour', 'vinecopulib_pyvinecopulib',\
           'elfi-dev_elfi', 'mattjj_pyhsmm']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,0042d1f5292516191796d6ba7d4dda28e0053a7f,falkonml_falkon


In [3]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

 24%|███████████████████▎                                                           | 726/2974 [12:20<38:11,  1.02s/it]

Got Errors {'0cce2907de97025e6d866674efe452158338c027': 'Key 0cce2907de97025e6d866674efe452158338c027 not found in /da5_fast/All.sha1c/commit_12.tch'}


 25%|███████████████████▋                                                           | 739/2974 [12:33<37:57,  1.02s/it]

Got Errors {'19d7909d4f47283a38056bd51712ea81823accd8': 'Key 19d7909d4f47283a38056bd51712ea81823accd8 not found in /da5_fast/All.sha1c/commit_25.tch'}


 25%|███████████████████▋                                                           | 740/2974 [12:34<37:57,  1.02s/it]

...

Got Errors {'76002fd878d2de7595ea94d0a825c6be81b6c771': 'Key 76002fd878d2de7595ea94d0a825c6be81b6c771 not found in /da5_fast/All.sha1c/commit_118.tch'}


 99%|█████████████████████████████████████████████████████████████████████████████ | 2936/2974 [49:53<00:38,  1.02s/it]

Got Errors {'c8059160d363b492241d86b4e76c35e4a5f138c3': 'Key c8059160d363b492241d86b4e76c35e4a5f138c3 not found in /da5_fast/All.sha1c/commit_72.tch'}


100%|██████████████████████████████████████████████████████████████████████████████| 2974/2974 [50:31<00:00,  1.02s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,0042d1f5292516191796d6ba7d4dda28e0053a7f,e4b6d1c6a7f61d527438b0814c38b388525a1348,[2885f7ae2dbdde5706ac9cc772e3cb005323626e],gmeanti <giacomo.meanti@gmail.com>,1643742599,+0100,gmeanti <giacomo.meanti@gmail.com>,1643742599,+0100,[skip ci] Update documentation,0042d1f5292516191796d6ba7d4dda28e0053a7f,falkonml_falkon
1,00447cc08941a87f8e7f9988434cd38934327e68,519778b0313f52bf6e77a27f60948bbeca19ea8e,[4f71f310ca457670b8ca44cdec82804248b9a06e],gmeanti <giacomo.meanti@gmail.com>,1618143109,+0200,gmeanti <giacomo.meanti@gmail.com>,1618143109,+0200,Update keops v1.5\n,00447cc08941a87f8e7f9988434cd38934327e68,falkonml_falkon


In [4]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '0042d1f5292516191796d6ba7d4dda28e0053a7f', 'tree': 'e4b6d1c6a7f61d527438b0814c38b388525a1348', 'parent': ['2885f7ae2dbdde5706ac9cc772e3cb005323626e'], 'author': 'gmeanti <giacomo.meanti@gmail.com>', 'author_time': 1643742599, 'author_tz': '+0100', 'committer': 'gmeanti <giacomo.meanti@gmail.com>', 'committer_time': 1643742599, 'committer_tz': '+0100', 'message': '[skip ci] Update documentation'}


In [9]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = df_commit_data[['project', 'commit', 'author', 'author_time', 'message']].copy()
dfinf.rename(columns={'author_time': 'time'}, inplace=True)

In [10]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,falkonml_falkon,0042d1f5292516191796d6ba7d4dda28e0053a7f,gmeanti <giacomo.meanti@gmail.com>,1643742599,[skip ci] Update documentation


In [11]:
#mode a means append, so you have all your projects in the same file
yournetid='mjohn326'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github